# 02 · Pelatihan U-Net dan DeepLabV3+ (T4-A, Raabin-WBC)

**Cara pakai (tiap anggota, di akun Colab masing-masing)**
1. Ketua sudah menaruh `data.py`, `runlog.py` (dan folder `splits/`) di folder Drive bersama. Taruh `metrics.py`, `models.py`, `train.py` di folder `code/` yang sama.
2. Runtime → Change runtime type → **T4 GPU**.
3. Isi `OWNER` di sel setup dengan **satu nama tetap** (mis. nama depan). Nama ini menentukan file log `logs/log_<OWNER>.csv`.
4. Jalankan sel setup, lalu **sel uji cepat**. Setelah itu jalankan hanya sel milikmu.

**Aturan**
- Satu faktor berubah per perbandingan, sisanya tetap pada baseline (U-Net ResNet-34, CE+Dice, split `grouped`, augmentasi warna aktif, 256 px, seed 42).
- Semua pemilihan epoch memakai **validasi**. `eval_test=True` hanya untuk konfigurasi final, sekali, di akhir.
- Jangan mengubah file di `splits/`.
- Run yang gagal tetap tercatat otomatis. Run yang terputus karena sesi Colab habis TIDAK tercatat, catat manual.

In [2]:
# ===== SEL SETUP (jalankan tiap sesi Colab baru) =====
from google.colab import drive; drive.mount('/content/drive')
!apt-get install -y unrar > /dev/null 2>&1
!pip -q install albumentations segmentation-models-pytorch
DRIVE = "/content/drive/MyDrive/PROYEK UAS DEEP LEARNING"

!mkdir -p /content/raabin_gt/bmp
!unrar x -inul "{DRIVE}/GroundTruths_bmpformat.rar" /content/raabin_gt/bmp/

import sys; sys.path.append(f"{DRIVE}/code")
import warnings; warnings.filterwarnings("ignore")
from train import run_experiment, run_seeds

OWNER = "Alpin"      # satu nama tetap, tanpa spasi
assert OWNER and " " not in OWNER, "OWNER tidak boleh kosong atau mengandung spasi"
print("OWNER =", OWNER)

Mounted at /content/drive
OWNER = Alpin


## Uji cepat (1 epoch)
Memastikan GPU, unduhan bobot pralatih, dan penulisan log berjalan. Tercatat di log dengan `ablation='smoke_test'`; baris ini dibuang saat laporan. Perkiraan waktu 1-3 menit (belum diukur di T4).

In [3]:
run_experiment(OWNER, "smoke_test", "-", epochs=1)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

ep  1/1 | loss 0.7026 | val dice cyto 0.8203 nuc 0.9613 | skor 0.8908


{'val_dice_bg': 0.9871,
 'val_iou_bg': 0.9749,
 'val_dice_cyto': 0.8203,
 'val_dice_cyto_sd': 0.1209,
 'val_iou_cyto': 0.7105,
 'val_dice_nuc': 0.9613,
 'val_dice_nuc_sd': 0.0225,
 'val_iou_nuc': 0.9263,
 'val_miou': 0.8706,
 'val_miou_sd': 0.0543,
 'val_hd95_cyto': 21.1046,
 'val_hd95_cyto_sd': 22.8182,
 'val_hd95_nuc': 7.5755,
 'val_hd95_nuc_sd': 12.0426,
 'run_id': 'Alpin_1123_071806',
 'best_epoch': 1}

## Baseline (dijalankan SEKALI untuk seluruh tim)
Baseline U-Net milik **Anggota 2**, baseline DeepLabV3+ milik **Anggota 3**. Semua ablasi dibandingkan dengan baseline U-Net ini. Jangan dijalankan ulang oleh anggota lain.

In [ ]:
# Baseline U-Net (Anggota 2)
run_experiment(OWNER, "baseline", "unet_ce_dice", arch="unet")

In [ ]:
# Baseline DeepLabV3+ (Anggota 3)
run_experiment(OWNER, "baseline", "deeplabv3plus_ce_dice", arch="deeplabv3plus")

## Ablasi per anggota (U-Net, 1 seed = 42, evaluasi di VALIDASI)
Nilai yang sama dengan baseline tidak dijalankan lagi. Jika memori GPU habis (terutama 384 px), turunkan `batch_size` ke 8 dan itu tercatat di log.

In [ ]:
# Anggota 1 (Ketua): skema split. Baseline memakai 'grouped'; ini menambah 'random' (bocor antar-apusan)
run_experiment(OWNER, "split", "random", split="random")

In [ ]:
# Anggota 2: fungsi loss. Baseline = ce_dice; tambahkan tiga lainnya
for l in ["ce", "dice", "focal_dice"]:
    run_experiment(OWNER, "loss", l, loss=l)

In [ ]:
# Anggota 2
run_experiment(OWNER, "loss", "ce_p20", loss="ce", patience=20)

In [ ]:
# Anggota 3: augmentasi warna. Baseline = aktif; ini menambah 'nonaktif'
run_experiment(OWNER, "color_aug", "off", color_aug=False)

In [ ]:
# Anggota 3
run_experiment(OWNER, "color_aug", "off_p20", color_aug=False, patience=20)

In [4]:
# Anggota 4: resolusi masukan. Baseline = 256; ini menambah 384
run_experiment(OWNER, "resolution", "384", img_size=384)

ep  1/20 | loss 0.6809 | val dice cyto 0.7933 nuc 0.9410 | skor 0.8671
ep  2/20 | loss 0.2223 | val dice cyto 0.8067 nuc 0.9425 | skor 0.8746
ep  3/20 | loss 0.1827 | val dice cyto 0.8238 nuc 0.9595 | skor 0.8917
ep  4/20 | loss 0.1584 | val dice cyto 0.8665 nuc 0.9700 | skor 0.9183
ep  5/20 | loss 0.1444 | val dice cyto 0.8633 nuc 0.9714 | skor 0.9173
ep  6/20 | loss 0.1354 | val dice cyto 0.8753 nuc 0.9687 | skor 0.9220
ep  7/20 | loss 0.1336 | val dice cyto 0.8434 nuc 0.9677 | skor 0.9056
ep  8/20 | loss 0.1280 | val dice cyto 0.8791 nuc 0.9676 | skor 0.9234
ep  9/20 | loss 0.1159 | val dice cyto 0.8794 nuc 0.9703 | skor 0.9248
ep 10/20 | loss 0.1146 | val dice cyto 0.8829 nuc 0.9701 | skor 0.9265
ep 11/20 | loss 0.1114 | val dice cyto 0.8874 nuc 0.9698 | skor 0.9286
ep 12/20 | loss 0.1049 | val dice cyto 0.8904 nuc 0.9695 | skor 0.9300
ep 13/20 | loss 0.1006 | val dice cyto 0.8809 nuc 0.9679 | skor 0.9244
ep 14/20 | loss 0.0998 | val dice cyto 0.8847 nuc 0.9705 | skor 0.9276
ep 15/

{'val_dice_bg': 0.9946,
 'val_iou_bg': 0.9893,
 'val_dice_cyto': 0.8976,
 'val_dice_cyto_sd': 0.0814,
 'val_iou_cyto': 0.8224,
 'val_dice_nuc': 0.9725,
 'val_dice_nuc_sd': 0.0113,
 'val_iou_nuc': 0.9468,
 'val_miou': 0.9195,
 'val_miou_sd': 0.0387,
 'val_hd95_cyto': 9.3188,
 'val_hd95_cyto_sd': 16.3334,
 'val_hd95_nuc': 3.9502,
 'val_hd95_nuc_sd': 1.1898,
 'run_id': 'Alpin_1123_071856',
 'best_epoch': 19}

In [ ]:
# Opsional bila waktu cukup: encoder pralatih vs inisialisasi acak
run_experiment(OWNER, "pretrained", "random_init", pretrained=False)

## Perbandingan utama: 3 seed untuk dua konfigurasi terbaik
Pilih dua konfigurasi berdasarkan hasil **validasi** (misalnya U-Net vs DeepLabV3+ dengan pengaturan terbaik). `eval_test=True` menyentuh set uji, jadi jalankan ini **hanya setelah semua ablasi selesai dan konfigurasinya final**, sekali saja. `save_path` menyimpan bobot untuk model card (satu file ~100 MB).

In [ ]:
import pandas as pd
res = []
for s in (42, 43, 44):
    res.append(run_experiment(OWNER, "main", "unet_final", arch="unet", seed=s,
                              eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/unet_final_seed{s}.pt"))
df_unet = pd.DataFrame(res)
cols = [c for c in df_unet.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
print(pd.DataFrame({"mean": df_unet[cols].mean(), "sd": df_unet[cols].std(ddof=1)}).round(4))

In [ ]:
import pandas as pd
res = []
for s in (42, 43, 44):
    res.append(run_experiment(OWNER, "main", "deeplabv3plus_final", arch="deeplabv3plus", seed=s,
                              eval_test=True,
                              save_path=f"{DRIVE}/checkpoints/deeplabv3plus_final_seed{s}.pt"))
df_dlv3 = pd.DataFrame(res)
cols = [c for c in df_dlv3.columns if c.startswith(("val_", "test_")) and not c.endswith("_sd")]
print(pd.DataFrame({"mean": df_dlv3[cols].mean(), "sd": df_dlv3[cols].std(ddof=1)}).round(4))

## Gabung seluruh log (Ketua, di akhir)
Menghasilkan tabel log eksperimen, total jam GPU, dan bukti bahwa set uji hanya dipakai pada run final.

In [ ]:
import glob, pandas as pd
log = pd.concat([pd.read_csv(f) for f in glob.glob(f"{DRIVE}/logs/log_*.csv")], ignore_index=True)
log = log[log.ablation != "smoke_test"]
log.to_csv(f"{DRIVE}/logs/LOG_GABUNGAN.csv", index=False)
print("Total run:", len(log), "| gagal:", (log.status == "failed").sum())
print("Total jam GPU:", round(log.gpu_minutes.sum() / 60, 2))
print("Run yang memakai set uji:", int(log.used_test.sum()))
print(log.groupby("owner").gpu_minutes.sum().div(60).round(2))